In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import math

# Consistent styling
COLORS = {
    'gt': '#2ecc71',       
    'pos': '#e74c3c',      
    'neg': '#3498db',      
    'ignore': '#95a5a6',   
    'proposal': '#e67e22',  
    'detection': '#9b59b6'
}

# ImageNet de-normalization for display
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def denormalize(img_tensor):
    """Convert a normalized image tensor back to displayable [0,1] range."""
    img = img_tensor.cpu() * STD + MEAN
    return img.clamp(0, 1).permute(1, 2, 0).numpy()


# ============================================================================
#  (PROVIDED — do not modify)
# ============================================================================
# ============================================================================
#  DATASET SAMPLES
# ============================================================================

def viz_dataset_samples(dataset, n=8, save_path='viz_01_dataset_samples.png'):
    """
    Show sample images from the dataset with ground-truth bounding boxes.

    This is always the first thing to look at — understand your data before
    building a model. You should see pedestrians with green bounding boxes.
    """
    rows = 2
    cols = n // rows
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = axes.flatten()

    for i in range(n):
        img_tensor, boxes, labels = dataset[i]
        img = denormalize(img_tensor)
        ax = axes[i]
        ax.imshow(img)

        for box in boxes:
            x1, y1, x2, y2 = box.numpy()
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=COLORS['gt'], facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 3, 'pedestrian', fontsize=7, color='white',
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', fc=COLORS['gt'], alpha=0.8))

        ax.set_title(f'Image {i} ({len(boxes)} objects)', fontsize=9)
        ax.set_axis_off()

    plt.suptitle('Dataset Samples with Ground-Truth Boxes', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")


# ============================================================================
#  ANCHOR TEMPLATES (cell anchors at origin)
# ============================================================================

def viz_anchor_templates(save_path='viz_02_anchor_templates.png'):
    """
    Show all 9 anchor templates centered at the origin.

    This helps verify TODO 1a. Each anchor is a different (size, aspect_ratio)
    combination. You should see 3 groups (by size) × 3 shapes (by ratio).
    """
    gen = AnchorGenerator(sizes=[64, 128, 256], aspect_ratios=[0.5, 1.0, 2.0])
    cell_anchors = gen.generate_cell_anchors()

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    size_names = ['Small (64)', 'Medium (128)', 'Large (256)']
    ratio_names = ['Wide (0.5)', 'Square (1.0)', 'Tall (2.0)']
    ratio_colors = ['#e74c3c', '#2ecc71', '#3498db']

    for size_idx, (ax, size_name) in enumerate(zip(axes, size_names)):
        ax.set_xlim(-200, 200)
        ax.set_ylim(-200, 200)
        ax.set_aspect('equal')
        ax.axhline(y=0, color='#bdc3c7', linewidth=0.5, linestyle='--')
        ax.axvline(x=0, color='#bdc3c7', linewidth=0.5, linestyle='--')
        ax.set_title(size_name, fontsize=12, fontweight='bold')
        ax.set_xlabel('pixels')
        ax.set_ylabel('pixels')

        for ratio_idx in range(3):
            anchor_idx = size_idx * 3 + ratio_idx
            x1, y1, x2, y2 = cell_anchors[anchor_idx].numpy()
            w, h = x2 - x1, y2 - y1
            rect = patches.Rectangle(
                (x1, y1), w, h, linewidth=2.5,
                edgecolor=ratio_colors[ratio_idx],
                facecolor=ratio_colors[ratio_idx], alpha=0.15,
                label=f'{ratio_names[ratio_idx]}: {w:.0f}×{h:.0f}'
            )
            ax.add_patch(rect)

        ax.legend(fontsize=9, loc='upper right')

    plt.suptitle('Anchor Templates: 3 Sizes × 3 Aspect Ratios = 9 Anchors per Location',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")


# ============================================================================
#  ANCHOR GRID (tiled anchors across image)
# ============================================================================

def viz_anchor_grid(image_size=300, save_path='viz_03_anchor_grid.png'):
    """
    Show how anchors tile across the image at every feature map location.

    This helps verify TODO 1b. Left: all anchor centers on a grid (stride=16).
    Right: a subset of actual anchor boxes to show coverage.
    """
    gen = AnchorGenerator(sizes=[64, 128, 256], aspect_ratios=[0.5, 1.0, 2.0])
    stride = 16
    feat_h, feat_w = image_size // stride, image_size // stride
    anchors = gen.generate((feat_h, feat_w), stride)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Left: anchor centers
    ax = axes[0]
    cx = (anchors[:, 0] + anchors[:, 2]) / 2
    cy = (anchors[:, 1] + anchors[:, 3]) / 2
    # Each location has 9 anchors with same center, so take every 9th
    ax.scatter(cx[::9].numpy(), cy[::9].numpy(), s=8, c='#e74c3c', zorder=5)
    ax.set_xlim(0, image_size)
    ax.set_ylim(image_size, 0)
    ax.set_aspect('equal')
    ax.set_title(f'Anchor Centers\n({feat_h}×{feat_w} = {feat_h*feat_w} locations, stride={stride})',
                 fontsize=10)
    ax.set_xlabel('x (pixels)')
    ax.set_ylabel('y (pixels)')
    # Draw grid
    for i in range(feat_h + 1):
        ax.axhline(y=i * stride, color='#ecf0f1', linewidth=0.5)
    for j in range(feat_w + 1):
        ax.axvline(x=j * stride, color='#ecf0f1', linewidth=0.5)

    # Middle: square anchors only (ratio=1, size=128)
    ax = axes[1]
    ax.set_xlim(0, image_size)
    ax.set_ylim(image_size, 0)
    ax.set_aspect('equal')
    # Index 4 = size_idx=1 (128), ratio_idx=1 (1.0) → 1*3+1=4
    subset = anchors[4::9]
    for anchor in subset:
        x1, y1, x2, y2 = anchor.numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=0.3, edgecolor='#3498db', facecolor='#3498db', alpha=0.04
        )
        ax.add_patch(rect)
    ax.set_title(f'128×128 Anchors (ratio=1.0)\n{len(subset)} boxes', fontsize=10)

    # Right: all 9 anchors at one central location
    ax = axes[2]
    ax.set_xlim(0, image_size)
    ax.set_ylim(image_size, 0)
    ax.set_aspect('equal')
    # Get anchors at center of image
    center_row = feat_h // 2
    center_col = feat_w // 2
    center_idx = (center_row * feat_w + center_col) * 9
    center_anchors = anchors[center_idx:center_idx + 9]

    ratio_colors = ['#e74c3c', '#2ecc71', '#3498db']
    for i, anchor in enumerate(center_anchors):
        x1, y1, x2, y2 = anchor.numpy()
        color = ratio_colors[i % 3]
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor=color, alpha=0.1
        )
        ax.add_patch(rect)
    # Mark center
    ccx = (center_col * stride + stride / 2)
    ccy = (center_row * stride + stride / 2)
    ax.plot(ccx, ccy, 'k+', markersize=15, markeredgewidth=2)
    ax.set_title(f'All 9 Anchors at Center\n({ccx:.0f}, {ccy:.0f})', fontsize=10)

    plt.suptitle(f'Anchor Tiling: {len(anchors)} total anchors for {image_size}×{image_size} image',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")


# ============================================================================
#  ANCHOR-TARGET ASSIGNMENT
# ============================================================================

def viz_anchor_assignment(dataset, save_path='viz_04_anchor_assignment.png'):
    """
    Show how anchors are assigned as positive/negative/ignore relative to GT boxes.

    This visualizes TODO 3. For one image, we show:
      - Green: GT boxes
      - Red: positive anchors (IoU ≥ 0.7 with some GT)
      - Blue: a sample of negative anchors (IoU < 0.3 with all GT)
      - Gray: ignored anchors (in between)
    Also shows the IoU distribution histogram.
    """
    gen = AnchorGenerator(sizes=[64, 128, 256], aspect_ratios=[0.5, 1.0, 2.0])
    stride = 16
    image_size = dataset.image_size

    # Pick an image with multiple objects
    best_idx = 0
    best_count = 0
    for i in range(min(len(dataset), 50)):
        _, boxes, _ = dataset[i]
        if len(boxes) > best_count:
            best_count = len(boxes)
            best_idx = i

    img_tensor, gt_boxes, gt_labels = dataset[best_idx]
    img = denormalize(img_tensor)

    feat_h, feat_w = image_size // stride, image_size // stride
    anchors = gen.generate((feat_h, feat_w), stride)

    labels, reg_targets = rpn_assign_targets(anchors, gt_boxes)
    iou_matrix = compute_iou(anchors, gt_boxes)
    max_iou = iou_matrix.max(dim=1).values

    pos_mask = labels == 1
    neg_mask = labels == 0
    ign_mask = labels == -1

    fig = plt.figure(figsize=(18, 8))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.8])

    # Left: positive anchors overlaid on image
    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(img)
    for box in gt_boxes:
        x1, y1, x2, y2 = box.numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=3, edgecolor=COLORS['gt'], facecolor='none', linestyle='-'
        )
        ax1.add_patch(rect)

    pos_anchors = anchors[pos_mask]
    for anchor in pos_anchors:
        x1, y1, x2, y2 = anchor.numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1.5, edgecolor=COLORS['pos'], facecolor=COLORS['pos'], alpha=0.15
        )
        ax1.add_patch(rect)

    ax1.set_title(f'Positive Anchors ({pos_mask.sum()} anchors, IoU ≥ 0.7)\n'
                  f'+ GT boxes (green)', fontsize=10)
    ax1.set_axis_off()

    # Middle: sample of negative anchors
    ax2 = fig.add_subplot(gs[1])
    ax2.imshow(img)
    for box in gt_boxes:
        x1, y1, x2, y2 = box.numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=3, edgecolor=COLORS['gt'], facecolor='none'
        )
        ax2.add_patch(rect)

    # Show a random subset of negatives (there are thousands)
    neg_indices = torch.where(neg_mask)[0]
    sample_neg = neg_indices[torch.randperm(len(neg_indices))[:30]]
    for idx in sample_neg:
        x1, y1, x2, y2 = anchors[idx].numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=0.8, edgecolor=COLORS['neg'], facecolor=COLORS['neg'], alpha=0.08
        )
        ax2.add_patch(rect)

    ax2.set_title(f'Negative Anchors (30 of {neg_mask.sum()} shown, IoU < 0.3)\n'
                  f'+ GT boxes (green)', fontsize=10)
    ax2.set_axis_off()

    # Right: IoU histogram
    ax3 = fig.add_subplot(gs[2])
    iou_values = max_iou.numpy()
    ax3.hist(iou_values, bins=50, color='#95a5a6', edgecolor='white', alpha=0.7)
    ax3.axvline(x=0.3, color=COLORS['neg'], linewidth=2, linestyle='--', label='Neg thresh (0.3)')
    ax3.axvline(x=0.7, color=COLORS['pos'], linewidth=2, linestyle='--', label='Pos thresh (0.7)')

    # Shade regions
    ax3.axvspan(0, 0.3, alpha=0.1, color=COLORS['neg'])
    ax3.axvspan(0.3, 0.7, alpha=0.1, color=COLORS['ignore'])
    ax3.axvspan(0.7, 1.0, alpha=0.1, color=COLORS['pos'])

    ax3.set_xlabel('Max IoU with GT', fontsize=10)
    ax3.set_ylabel('Number of Anchors', fontsize=10)
    ax3.set_title('IoU Distribution\n'
                  f'Pos: {pos_mask.sum()}, Neg: {neg_mask.sum()}, Ign: {ign_mask.sum()}',
                  fontsize=10)
    ax3.legend(fontsize=8)

    plt.suptitle('Anchor-Target Assignment (TODO 3)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")


# ============================================================================
#  BACKBONE FEATURE MAPS
# ============================================================================

def viz_backbone_features(dataset, save_path='viz_05_backbone_features.png'):
    """
    Show what the ResNet-18 backbone sees.

    The feature map is 512-dimensional, so we show the mean activation and a
    few individual channels. High activations tend to correspond to object
    locations — this is the representation the RPN operates on.
    """
    backbone = Backbone()
    backbone.eval()

    img_tensor, gt_boxes, _ = dataset[0]
    img = denormalize(img_tensor)

    with torch.no_grad():
        features = backbone(img_tensor.unsqueeze(0))  # (1, 512, H, W)

    feat = features[0].numpy()  # (512, H, W)

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    # Top-left: original image with GT
    ax = axes[0, 0]
    ax.imshow(img)
    for box in gt_boxes:
        x1, y1, x2, y2 = box.numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=COLORS['gt'], facecolor='none'
        )
        ax.add_patch(rect)
    ax.set_title('Input Image + GT', fontsize=10)
    ax.set_axis_off()

    # Top-center: mean activation
    ax = axes[0, 1]
    mean_feat = feat.mean(axis=0)
    im = ax.imshow(mean_feat, cmap='hot', interpolation='bilinear')
    ax.set_title(f'Mean Feature Activation\n({feat.shape[1]}×{feat.shape[2]})', fontsize=10)
    ax.set_axis_off()
    plt.colorbar(im, ax=ax, fraction=0.046)

    # Top-right: max activation
    ax = axes[0, 2]
    max_feat = feat.max(axis=0)
    im = ax.imshow(max_feat, cmap='hot', interpolation='bilinear')
    ax.set_title('Max Feature Activation', fontsize=10)
    ax.set_axis_off()
    plt.colorbar(im, ax=ax, fraction=0.046)

    # Top-far-right: std of activations
    ax = axes[0, 3]
    std_feat = feat.std(axis=0)
    im = ax.imshow(std_feat, cmap='viridis', interpolation='bilinear')
    ax.set_title('Std of Activations\n(high = discriminative)', fontsize=10)
    ax.set_axis_off()
    plt.colorbar(im, ax=ax, fraction=0.046)

    # Bottom row: individual channels (pick 4 with highest variance = most interesting)
    channel_var = feat.var(axis=(1, 2))
    top_channels = np.argsort(channel_var)[-4:][::-1]

    for i, ch in enumerate(top_channels):
        ax = axes[1, i]
        im = ax.imshow(feat[ch], cmap='RdBu_r', interpolation='bilinear')
        ax.set_title(f'Channel {ch}\n(var={channel_var[ch]:.3f})', fontsize=9)
        ax.set_axis_off()
        plt.colorbar(im, ax=ax, fraction=0.046)

    plt.suptitle(f'Backbone Feature Maps: {feat.shape[0]} channels, '
                 f'{feat.shape[1]}×{feat.shape[2]} spatial '
                 f'(stride {dataset.image_size // feat.shape[1]})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")


# ============================================================================
#  RPN PREDICTIONS (objectness + box deltas)
# ============================================================================

def viz_rpn_predictions(dataset, save_path='viz_06_rpn_predictions.png'):
    """
    Show raw RPN outputs before NMS: objectness heatmap and top-scoring anchors.

    Left: objectness score heatmap (max over anchors at each location).
    Right: top-50 highest-scoring anchors drawn on the image.

    Before training these will be random; after training, high-objectness
    regions should correspond to pedestrian locations.
    """
    model = SimpleFasterRCNN(num_classes=2, image_size=300)
    model.eval()

    img_tensor, gt_boxes, _ = dataset[0]
    img = denormalize(img_tensor)

    with torch.no_grad():
        features = model.backbone(img_tensor.unsqueeze(0))
        feat_h, feat_w = features.shape[2], features.shape[3]
        objectness, box_deltas = model.rpn_head(features)
        anchors = model.anchor_generator.generate(
            (feat_h, feat_w), model.backbone.stride
        )

    # objectness: (1, H*W*K, 2) — softmax to get probabilities
    obj_scores = F.softmax(objectness[0], dim=1)[:, 1]  # P(object)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Left: objectness heatmap
    ax = axes[0]
    # Reshape scores to (H, W, K) and take max over K anchors per location
    scores_map = obj_scores.reshape(feat_h, feat_w, 9).max(dim=2).values.numpy()
    ax.imshow(img)
    ax.imshow(
        np.kron(scores_map, np.ones((model.backbone.stride, model.backbone.stride))),
        alpha=0.5, cmap='hot',
        extent=[0, img.shape[1], img.shape[0], 0]
    )
    for box in gt_boxes:
        x1, y1, x2, y2 = box.numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=COLORS['gt'], facecolor='none'
        )
        ax.add_patch(rect)
    ax.set_title('Objectness Heatmap\n(max score per location)', fontsize=10)
    ax.set_axis_off()

    # Middle: top-50 scoring raw anchors
    ax = axes[1]
    ax.imshow(img)
    top_k = min(50, len(obj_scores))
    top_indices = obj_scores.topk(top_k).indices
    for idx in top_indices:
        x1, y1, x2, y2 = anchors[idx].numpy()
        score = obj_scores[idx].item()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1, edgecolor=COLORS['proposal'],
            facecolor=COLORS['proposal'], alpha=max(0.05, score * 0.5)
        )
        ax.add_patch(rect)
    for box in gt_boxes:
        x1, y1, x2, y2 = box.numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=COLORS['gt'], facecolor='none'
        )
        ax.add_patch(rect)
    ax.set_title(f'Top {top_k} Scoring Anchors (before NMS)\n'
                 f'(opacity ∝ score)', fontsize=10)
    ax.set_axis_off()

    # Right: score distribution
    ax = axes[2]
    ax.hist(obj_scores.numpy(), bins=80, color='#e67e22', edgecolor='white', alpha=0.7)
    ax.set_xlabel('Objectness Score P(object)', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(f'Score Distribution\n({len(anchors)} anchors)', fontsize=10)
    ax.set_yscale('log')

    plt.suptitle('RPN Raw Predictions (before training — expect random)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")


# ============================================================================
#  REGION PROPOSALS (after NMS)
# ============================================================================

def viz_proposals(dataset, device, model=None, save_path='viz_07_proposals.png'):
    """
    Show region proposals after NMS for multiple images.

    These are the boxes that get passed to the second stage (RoI Pool +
    Detection Head). Good proposals should cover the objects tightly.
    Before training, proposals will be scattered randomly.
    """
    if model is None:
        model = SimpleFasterRCNN(num_classes=2, image_size=300)
    model.eval()
    model.to(device)

    n_images = 4
    fig, axes = plt.subplots(2, n_images, figsize=(5 * n_images, 10))

    for i in range(n_images):
        img_tensor, gt_boxes, _ = dataset[i]
        img = denormalize(img_tensor)

        with torch.no_grad():
            output = model(img_tensor.unsqueeze(0).to(device))
        proposals = output['proposals'][0]

        # Top row: all proposals
        ax = axes[0, i]
        ax.imshow(img)
        for j, prop in enumerate(proposals[:100]):
            x1, y1, x2, y2 = prop.cpu().numpy()
            alpha = max(0.02, 1.0 - j / 100)  # fade out lower-ranked
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=0.5, edgecolor=COLORS['proposal'],
                facecolor=COLORS['proposal'], alpha=alpha * 0.15
            )
            ax.add_patch(rect)
        for box in gt_boxes:
            x1, y1, x2, y2 = box.numpy()
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2.5, edgecolor=COLORS['gt'], facecolor='none'
            )
            ax.add_patch(rect)
        ax.set_title(f'{len(proposals)} proposals (top 100 shown)', fontsize=9)
        ax.set_axis_off()

        # Bottom row: top-10 proposals only
        ax = axes[1, i]
        ax.imshow(img)
        for j, prop in enumerate(proposals[:10]):
            x1, y1, x2, y2 = prop.cpu().numpy()
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=COLORS['proposal'], facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 3, f'#{j+1}', fontsize=7, color='white',
                    bbox=dict(boxstyle='round,pad=0.15', fc=COLORS['proposal'], alpha=0.8))
        for box in gt_boxes:
            x1, y1, x2, y2 = box.numpy()
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2.5, edgecolor=COLORS['gt'], facecolor='none'
            )
            ax.add_patch(rect)
        ax.set_title('Top 10 proposals', fontsize=9)
        ax.set_axis_off()

    axes[0, 0].set_ylabel('All proposals\n(top 100)', fontsize=11, labelpad=10)
    axes[1, 0].set_ylabel('Top 10', fontsize=11, labelpad=10)

    plt.suptitle('Region Proposals After NMS (green = GT, orange = proposals)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")



# ============================================================================
#  DETECTION HEAD OUTPUT (before and after training)
# ============================================================================

def viz_detections(dataset,device, model=None, save_path='viz_09_detections.png',
                   score_thresh=0.3):
    """
    Show final detection results: class predictions + refined boxes.

    For each proposal, the detection head predicts class scores and box
    refinements. We show proposals colored by their predicted class, with
    confidence scores. Before training everything will be near-random.
    """
    if model is None:
        model = SimpleFasterRCNN(num_classes=2, image_size=300)
    model.eval()

    n_images = 4
    fig, axes = plt.subplots(1, n_images, figsize=(5 * n_images, 5))

    class_names = PennFudanDataset.CLASSES

    for i in range(n_images):
        img_tensor, gt_boxes, _ = dataset[i]
        img = denormalize(img_tensor)
        ax = axes[i]
        ax.imshow(img)

        with torch.no_grad():
            output = model(img_tensor.unsqueeze(0).to(device))

        proposals = output['proposals'][0]
        logits = output['class_logits']
        probs = F.softmax(logits, dim=1)

        # Get per-proposal predictions
        scores, pred_classes = probs.max(dim=1)

        # Filter to non-background with decent score
        fg_mask = (pred_classes > 0) & (scores > score_thresh)
        fg_indices = torch.where(fg_mask)[0]

        # Split proposals per image (all belong to image 0 here)
        n_props = len(proposals)
        img_fg = fg_indices[fg_indices < n_props]

        n_det = 0
        for idx in img_fg:
            if idx >= len(proposals):
                continue
            x1, y1, x2, y2 = proposals[idx].cpu().numpy()
            cls = pred_classes[idx].item()
            score = scores[idx].item()
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=COLORS['detection'], facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 3,
                    f'{class_names[cls]}: {score:.2f}', fontsize=7, color='white',
                    bbox=dict(boxstyle='round,pad=0.2', fc=COLORS['detection'], alpha=0.8))
            n_det += 1

        # GT boxes
        for box in gt_boxes:
            bx1, by1, bx2, by2 = box.numpy()
            rect = patches.Rectangle(
                (bx1, by1), bx2 - bx1, by2 - by1,
                linewidth=2.5, edgecolor=COLORS['gt'], facecolor='none', linestyle='--'
            )
            ax.add_patch(rect)

        ax.set_title(f'{n_det} detections (thresh={score_thresh})', fontsize=9)
        ax.set_axis_off()

    legend_elements = [
        Line2D([0], [0], color=COLORS['gt'], linewidth=2, linestyle='--', label='Ground Truth'),
        Line2D([0], [0], color=COLORS['detection'], linewidth=2, label='Detection'),
    ]
    axes[-1].legend(handles=legend_elements, fontsize=8, loc='lower right')

    plt.suptitle('Detection Results (purple = prediction, green dashed = GT)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")


# ============================================================================
#  10. FULL PIPELINE OVERVIEW
# ============================================================================

def viz_pipeline_overview(dataset, device, save_path='viz_10_pipeline_overview.png'):
    """
    A single figure showing all stages of Faster R-CNN on one image:
      Input → Backbone Features → Objectness Heatmap → Proposals → Detections

    """
    model = SimpleFasterRCNN(num_classes=2, image_size=300)
    model.eval()

    backbone = model.backbone

    img_tensor, gt_boxes, _ = dataset[0]
    img = denormalize(img_tensor)

    with torch.no_grad():
        features = backbone(img_tensor.unsqueeze(0))
        feat_h, feat_w = features.shape[2], features.shape[3]
        objectness, box_deltas = model.rpn_head(features)
        anchors = model.anchor_generator.generate(
            (feat_h, feat_w), backbone.stride
        )
        output = model(img_tensor.unsqueeze(0))

    proposals = output['proposals'][0]
    obj_scores = F.softmax(objectness[0], dim=1)[:, 1]

    fig, axes = plt.subplots(1, 5, figsize=(25, 5))

    # 1. Input
    ax = axes[0]
    ax.imshow(img)
    for box in gt_boxes:
        x1, y1, x2, y2 = box.numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=COLORS['gt'], facecolor='none'
        )
        ax.add_patch(rect)
    ax.set_title('① Input Image\n+ Ground Truth', fontsize=10)
    ax.set_axis_off()

    # 2. Feature map
    ax = axes[1]
    feat = features[0].mean(dim=0).numpy()
    ax.imshow(feat, cmap='hot', interpolation='bilinear')
    ax.set_title(f'② Backbone Features\n({features.shape[1]}ch, {feat_h}×{feat_w})',
                 fontsize=10)
    ax.set_axis_off()

    # 3. Objectness heatmap
    ax = axes[2]
    scores_map = obj_scores.reshape(feat_h, feat_w, 9).max(dim=2).values.numpy()
    ax.imshow(img)
    ax.imshow(
        np.kron(scores_map, np.ones((backbone.stride, backbone.stride))),
        alpha=0.6, cmap='hot',
        extent=[0, img.shape[1], img.shape[0], 0]
    )
    ax.set_title('③ RPN Objectness\n(heatmap overlay)', fontsize=10)
    ax.set_axis_off()

    # 4. Proposals
    ax = axes[3]
    ax.imshow(img)
    for j, prop in enumerate(proposals[:20]):
        x1, y1, x2, y2 = prop.numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1.5, edgecolor=COLORS['proposal'], facecolor='none'
        )
        ax.add_patch(rect)
    for box in gt_boxes:
        x1, y1, x2, y2 = box.numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=COLORS['gt'], facecolor='none'
        )
        ax.add_patch(rect)
    ax.set_title(f'④ Proposals (top 20)\n({len(proposals)} total after NMS)', fontsize=10)
    ax.set_axis_off()

    # 5. Detections
    ax = axes[4]
    ax.imshow(img)
    logits = output['class_logits']
    probs = F.softmax(logits, dim=1)
    scores, pred_cls = probs.max(dim=1)
    fg = (pred_cls > 0) & (scores > 0.3)
    fg_idx = torch.where(fg)[0]
    for idx in fg_idx:
        if idx >= len(proposals):
            continue
        x1, y1, x2, y2 = proposals[idx].numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=COLORS['detection'], facecolor='none'
        )
        ax.add_patch(rect)
    for box in gt_boxes:
        x1, y1, x2, y2 = box.numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=COLORS['gt'], facecolor='none', linestyle='--'
        )
        ax.add_patch(rect)
    ax.set_title(f'⑤ Final Detections\n({fg.sum()} foreground)', fontsize=10)
    ax.set_axis_off()

    # Arrows between panels
    for i in range(4):
        fig.text((i + 1) * 0.2 + 0.005, 0.5, '→', fontsize=24,
                 ha='center', va='center', color='#7f8c8d', fontweight='bold')

    plt.suptitle('Faster R-CNN Pipeline: Image → Features → Objectness → Proposals → Detections',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")




In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.ops import RoIPool, nms
from torch.utils.data import DataLoader, Dataset
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict
import os
import re
import zipfile
import urllib.request
import warnings
warnings.filterwarnings("ignore")


# ============================================================================
#  UTILITIES (PROVIDED — do not modify)
# ============================================================================

def compute_iou(boxes1, boxes2):
    """
    Compute pairwise IoU between two sets of boxes.
    Args:
        boxes1: (N, 4) in [x1, y1, x2, y2]
        boxes2: (M, 4) in [x1, y1, x2, y2]
    Returns:
        iou: (N, M) pairwise IoU matrix
    """
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])

    inter_x1 = torch.max(boxes1[:, None, 0], boxes2[None, :, 0])
    inter_y1 = torch.max(boxes1[:, None, 1], boxes2[None, :, 1])
    inter_x2 = torch.min(boxes1[:, None, 2], boxes2[None, :, 2])
    inter_y2 = torch.min(boxes1[:, None, 3], boxes2[None, :, 3])

    inter_area = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
    union_area = area1[:, None] + area2[None, :] - inter_area
    return inter_area / (union_area + 1e-6)


def encode_boxes(anchors, gt_boxes):
    """
    Encode ground-truth boxes as offsets relative to anchors.
    Uses the standard (tx, ty, tw, th) parameterization from the R-CNN papers.

    tx = (gx - ax) / aw,  ty = (gy - ay) / ah
    tw = log(gw / aw),    th = log(gh / ah)

    where (ax, ay, aw, ah) is the anchor center and size, and similarly for gt.
    """
    a_cx = (anchors[:, 0] + anchors[:, 2]) / 2
    a_cy = (anchors[:, 1] + anchors[:, 3]) / 2
    a_w = anchors[:, 2] - anchors[:, 0]
    a_h = anchors[:, 3] - anchors[:, 1]

    g_cx = (gt_boxes[:, 0] + gt_boxes[:, 2]) / 2
    g_cy = (gt_boxes[:, 1] + gt_boxes[:, 3]) / 2
    g_w = gt_boxes[:, 2] - gt_boxes[:, 0]
    g_h = gt_boxes[:, 3] - gt_boxes[:, 1]

    tx = (g_cx - a_cx) / (a_w + 1e-6)
    ty = (g_cy - a_cy) / (a_h + 1e-6)
    tw = torch.log(g_w / (a_w + 1e-6) + 1e-6)
    th = torch.log(g_h / (a_h + 1e-6) + 1e-6)

    return torch.stack([tx, ty, tw, th], dim=1)


def decode_boxes(anchors, deltas):
    """
    Decode predicted box deltas back to [x1, y1, x2, y2] coordinates.
    Inverse of encode_boxes.
    """
    a_cx = (anchors[:, 0] + anchors[:, 2]) / 2
    a_cy = (anchors[:, 1] + anchors[:, 3]) / 2
    a_w = anchors[:, 2] - anchors[:, 0]
    a_h = anchors[:, 3] - anchors[:, 1]

    pred_cx = deltas[:, 0] * a_w + a_cx
    pred_cy = deltas[:, 1] * a_h + a_cy
    pred_w = torch.exp(deltas[:, 2].clamp(max=4.0)) * a_w
    pred_h = torch.exp(deltas[:, 3].clamp(max=4.0)) * a_h

    x1 = pred_cx - pred_w / 2
    y1 = pred_cy - pred_h / 2
    x2 = pred_cx + pred_w / 2
    y2 = pred_cy + pred_h / 2

    return torch.stack([x1, y1, x2, y2], dim=1)


def clip_boxes(boxes, image_size):
    """Clip boxes to image boundaries."""
    boxes[:, 0::2] = boxes[:, 0::2].clamp(0, image_size)
    boxes[:, 1::2] = boxes[:, 1::2].clamp(0, image_size)
    return boxes


def filter_small_boxes(boxes, min_size=4):
    """Remove boxes smaller than min_size."""
    w = boxes[:, 2] - boxes[:, 0]
    h = boxes[:, 3] - boxes[:, 1]
    keep = (w >= min_size) & (h >= min_size)
    return keep


In [ ]:
# ============================================================================
#  DATASET 
# ============================================================================

class PennFudanDataset(Dataset):
    """
    Penn-Fudan Pedestrian Detection dataset.

    Classes: 0 = background, 1 = pedestrian
    Download: automatic from UPenn servers on first use.

    If automatic download fails on your machine/cluster, manually download:
        https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip
    and extract so that data_root/PennFudanPed/PNGImages/ exists.
    """

    CLASSES = ['background', 'pedestrian']
    NUM_CLASSES = 2  # background + pedestrian

    DOWNLOAD_URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"

    def __init__(self, data_root='./data', image_size=300, download=True):
        self.image_size = image_size
        self.root = os.path.join(data_root, 'PennFudanPed')

        if download and not os.path.isdir(self.root):
            self._download(data_root)

        if not os.path.isdir(self.root):
            raise RuntimeError(
                f"Dataset not found at {self.root}. "
                f"Download PennFudanPed.zip from:\n"
                f"  {self.DOWNLOAD_URL}\n"
                f"and extract into {data_root}/"
            )

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        # List all images
        img_dir = os.path.join(self.root, 'PNGImages')
        self.image_files = sorted([
            os.path.join(img_dir, f)
            for f in os.listdir(img_dir) if f.endswith('.png')
        ])

        # Parse annotations
        ann_dir = os.path.join(self.root, 'Annotation')
        self.annotation_files = sorted([
            os.path.join(ann_dir, f)
            for f in os.listdir(ann_dir) if f.endswith('.txt')
        ])

        assert len(self.image_files) == len(self.annotation_files), \
            f"Mismatch: {len(self.image_files)} images vs {len(self.annotation_files)} annotations"

        print(f"PennFudanDataset: {len(self.image_files)} images, "
              f"class: pedestrian, image_size: {image_size}x{image_size}")

    def _download(self, data_root):
        """Download and extract the dataset."""
        os.makedirs(data_root, exist_ok=True)
        zip_path = os.path.join(data_root, 'PennFudanPed.zip')
        print(f"Downloading Penn-Fudan dataset to {zip_path}...")
        try:
            urllib.request.urlretrieve(self.DOWNLOAD_URL, zip_path)
            print("Extracting...")
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(data_root)
            os.remove(zip_path)
            print("Done.")
        except Exception as e:
            print(f"Automatic download failed: {e}")
            print(f"Please manually download from:\n  {self.DOWNLOAD_URL}")
            print(f"and extract into: {data_root}/")
            raise

    def _parse_annotation(self, ann_path):
        """Parse a Penn-Fudan annotation file to extract bounding boxes."""
        boxes = []
        with open(ann_path, 'r') as f:
            text = f.read()
        # Bounding boxes are in format: (Xmin, Ymin) - (Xmax, Ymax)
        for match in re.finditer(
            r'Bounding box for object \d+ .+ : \((\d+), (\d+)\) - \((\d+), (\d+)\)',
            text
        ):
            x1, y1, x2, y2 = [int(v) for v in match.groups()]
            boxes.append([x1, y1, x2, y2])
        return boxes

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img = Image.open(self.image_files[idx]).convert('RGB')
        orig_w, orig_h = img.size

        raw_boxes = self._parse_annotation(self.annotation_files[idx])

        boxes, labels = [], []
        for (x1, y1, x2, y2) in raw_boxes:
            # Scale to target image size
            boxes.append([
                x1 / orig_w * self.image_size,
                y1 / orig_h * self.image_size,
                x2 / orig_w * self.image_size,
                y2 / orig_h * self.image_size,
            ])
            labels.append(1)  # all pedestrians

        img = self.transform(img)
        boxes = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4))
        labels = torch.tensor(labels, dtype=torch.long) if labels else torch.zeros(0, dtype=torch.long)
        return img, boxes, labels


def collate_fn(batch):
    images = torch.stack([b[0] for b in batch])
    boxes = [b[1] for b in batch]
    labels = [b[2] for b in batch]
    return images, boxes, labels

dataset = PennFudanDataset()

## Task 3.1: Anchor Generation

In [ ]:
class AnchorGenerator:
    """
    Generates a dense grid of anchor boxes over a feature map.

    BACKGROUND:
    Before Faster R-CNN, detectors used Selective Search to propose ~2000
    candidate regions per image. This was slow (~2s per image) and ran on
    CPU outside the network. Faster R-CNN's key insight was: we can learn
    to propose regions using the same CNN features we already computed.

    Anchors are the mechanism for this. At each spatial location of the
    feature map, we place K reference boxes of different sizes and aspect
    ratios. The RPN then predicts, for each anchor:
      (a) Is there an object here? (objectness score)
      (b) How should I shift/resize this anchor to better fit the object?

    For a feature map of size H x W, we get H * W * K total anchors.
    """

    def __init__(self,
                 sizes=[64, 128, 256],
                 aspect_ratios=[0.5, 1.0, 2.0]):
        self.sizes = sizes
        self.aspect_ratios = aspect_ratios
        self.num_anchors_per_location = len(sizes) * len(aspect_ratios)

    def generate_cell_anchors(self):
        """
        TODO 1a: Generate anchor templates centered at the origin.

        For each combination of (size, aspect_ratio):
          - width  = size * sqrt(aspect_ratio)
          - height = size / sqrt(aspect_ratio)
          - anchor = [-w/2, -h/2, w/2, h/2]   (centered at 0,0)

        Note: "aspect_ratio" here is width/height.
        """
        anchors = []
        #### TODO ---- START YOUR CODE ----
        for size in self.sizes:
            for ratio in self.aspect_ratios:
                w = size * (ratio ** 0.5)      # width  = size * sqrt(ratio)
                h = size / (ratio ** 0.5)      # height = size / sqrt(ratio)
                anchors.append([-w/2, -h/2, w/2, h/2])
        #### TODO ---- END YOUR CODE ----
        return torch.tensor(anchors, dtype=torch.float32)

    def generate(self, feature_map_size, stride, device='cpu'):
        """
        TODO 1b: Tile cell anchors across the entire feature map.

        Center of cell (row, col) in image coordinates:
            center_x = col * stride + stride / 2
            center_y = row * stride + stride / 2
        """
        H, W = feature_map_size
        cell_anchors = self.generate_cell_anchors().to(device)  # (K, 4)
        K = cell_anchors.shape[0]

        ### TODO ---- START YOUR CODE ----
        # Grid of cell centres in image-pixel space
        shifts_x = (torch.arange(W, device=device) + 0.5) * stride
        shifts_y = (torch.arange(H, device=device) + 0.5) * stride

        # meshgrid → (H, W) grids; stack into (H*W, 4) shift vector [dx,dy,dx,dy]
        grid_y, grid_x = torch.meshgrid(shifts_y, shifts_x, indexing='ij')
        shifts = torch.stack(
            [grid_x.reshape(-1), grid_y.reshape(-1),
             grid_x.reshape(-1), grid_y.reshape(-1)], dim=1   # (H*W, 4)
        )

        # Broadcasting: (H*W, 1, 4) + (1, K, 4) → (H*W, K, 4)
        all_anchors = shifts[:, None, :] + cell_anchors[None, :, :]
        ### TODO ---- END YOUR CODE ----

        # Reshape to (H*W*K, 4)
        return all_anchors.reshape(-1, 4)

In [ ]:
print("Dataset samples...")
viz_dataset_samples(dataset)

print("Anchor templates...")
viz_anchor_templates()

print("Anchor grid tiling...")
viz_anchor_grid()



## Task 3.2: Region Proposal Network — Architecture 

In [ ]:
class RPNHead(nn.Module):
    """
    The Region Proposal Network head.

    Architecture:
        feature_map -> 3x3 conv (shared) -> ReLU
                                              |
                                 +------------+------------+
                                 |                         |
                             1x1 conv                  1x1 conv
                          (K*2 outputs)            (K*4 outputs)
                          objectness scores        box deltas
    """

    def __init__(self, in_channels=512, num_anchors=9):
        super().__init__()
        self.num_anchors = num_anchors
        ## TODO
        # ---- START YOUR CODE ----
        # Shared 3×3 conv (keeps spatial dims)
        self.conv          = nn.Conv2d(in_channels, in_channels, 3, padding=1)
        # 1×1 objectness head: 2 scores per anchor (background / object)
        self.objectness    = nn.Conv2d(in_channels, num_anchors * 2, 1)
        # 1×1 regression head: 4 deltas per anchor (tx, ty, tw, th)
        self.box_regression = nn.Conv2d(in_channels, num_anchors * 4, 1)

        # Weight initialisation (standard Faster R-CNN practice)
        for layer in [self.conv, self.objectness, self.box_regression]:
            nn.init.normal_(layer.weight, std=0.01)
            nn.init.zeros_(layer.bias)
        # ---- END YOUR CODE ----

    def forward(self, feature_map):
        """
        Args:
            feature_map: (B, C, H, W) from backbone

        Returns:
            objectness: (B, H*W*K, 2) objectness logits [background, object]
            box_deltas: (B, H*W*K, 4) predicted box offsets [tx, ty, tw, th]
        """
        ## TODO
        # ---- START YOUR CODE ----
        B, C, H, W = feature_map.shape
        K = self.num_anchors

        # Shared feature aggregation
        x = F.relu(self.conv(feature_map))          # (B, C, H, W)

        # Objectness: (B, K*2, H, W) → permute → (B, H, W, K, 2) → reshape → (B, H*W*K, 2)
        obj = self.objectness(x)                    # (B, K*2, H, W)
        obj = obj.permute(0, 2, 3, 1)              # (B, H, W, K*2)
        obj = obj.reshape(B, H * W * K, 2)         # (B, H*W*K, 2)

        # Box deltas: (B, K*4, H, W) → (B, H*W*K, 4)
        deltas = self.box_regression(x)             # (B, K*4, H, W)
        deltas = deltas.permute(0, 2, 3, 1)        # (B, H, W, K*4)
        deltas = deltas.reshape(B, H * W * K, 4)   # (B, H*W*K, 4)

        return obj, deltas
        # ---- END YOUR CODE ----

## Task 3.3: RPN Target Assignment and Loss

In [ ]:

def rpn_assign_targets(anchors, gt_boxes, pos_iou_thresh=0.7, neg_iou_thresh=0.3):
    """
    Assign ground-truth targets to anchors for RPN training.

    Returns:
        labels: (A,) — 1 for positive, 0 for negative, -1 for ignore
        regression_targets: (A, 4) — encoded box offsets (zeros for non-positive)
    """
    A = anchors.shape[0]
    device = anchors.device

    labels = torch.full((A,), -1, dtype=torch.long, device=device)
    regression_targets = torch.zeros((A, 4), dtype=torch.float32, device=device)

    if gt_boxes.shape[0] == 0:
        labels[:] = 0  # All negative if no GT
        return labels, regression_targets

    #### TODO
    # ---- START YOUR CODE ----
    # Step 1: Compute IoU matrix (A, G)
    iou_matrix = compute_iou(anchors, gt_boxes.to(device))   # (A, G)

    # Step 2: For each anchor, find its best-matching GT box and max IoU
    max_iou, matched_gt_idx = iou_matrix.max(dim=1)          # (A,)

    # Step 3: Negative — max IoU below threshold
    labels[max_iou < neg_iou_thresh] = 0

    # Step 4: Positive — max IoU above threshold
    labels[max_iou >= pos_iou_thresh] = 1

    # Step 5: For each GT box, force the anchor with the highest IoU to be positive
    #         (ensures every GT box has at least one positive anchor)
    best_anchor_per_gt = iou_matrix.argmax(dim=0)   # (G,)
    labels[best_anchor_per_gt] = 1

    # Step 6: Compute regression targets for ALL anchors (only positives used in loss)
    matched_gt_boxes = gt_boxes[matched_gt_idx]      # (A, 4)
    regression_targets = encode_boxes(anchors, matched_gt_boxes.to(device))
    # ---- END YOUR CODE ----

    return labels, regression_targets


def rpn_loss(objectness_logits, box_deltas, anchors, gt_boxes,
             num_samples=256, pos_fraction=0.5):
    """
    Compute RPN loss with balanced sampling (PROVIDED).
    """
    labels, reg_targets = rpn_assign_targets(anchors, gt_boxes)

    # Balanced sampling
    pos_idx = torch.where(labels == 1)[0]
    neg_idx = torch.where(labels == 0)[0]
    num_pos = min(int(num_samples * pos_fraction), len(pos_idx))
    num_neg = min(num_samples - num_pos, len(neg_idx))

    if len(pos_idx) > num_pos:
        pos_idx = pos_idx[torch.randperm(len(pos_idx))[:num_pos]]
    if len(neg_idx) > num_neg:
        neg_idx = neg_idx[torch.randperm(len(neg_idx))[:num_neg]]

    sampled_idx    = torch.cat([pos_idx, neg_idx])
    sampled_labels = labels[sampled_idx]

    cls_loss = F.cross_entropy(objectness_logits[sampled_idx], sampled_labels)

    if num_pos > 0:
        reg_loss = F.smooth_l1_loss(
            box_deltas[pos_idx], reg_targets[pos_idx], beta=1.0/9)
    else:
        reg_loss = torch.tensor(0.0, device=anchors.device)

    return cls_loss, reg_loss

## Task 3.4: ROI Pooling

In [ ]:
class RoIPoolingLayer(nn.Module):
    """
    Region of Interest Pooling.

    Divides each proposal's feature region into a fixed grid (output_size × output_size)
    and max-pools within each grid cell, producing fixed-size outputs regardless of
    proposal size.
    """

    def __init__(self, output_size=7, spatial_scale=1.0/16):
        super().__init__()
        self.output_size = output_size
        ########## TODO
        # ---- START YOUR CODE ----
        # torchvision RoIPool: maps image-coordinate proposals → feature-map coords
        # spatial_scale = 1/stride (e.g. 1/16 for backbone stride 16)
        from torchvision.ops import RoIPool
        self.roi_pool = RoIPool(output_size=(output_size, output_size),
                                spatial_scale=spatial_scale)
        # ---- END YOUR CODE ----

    def forward(self, feature_map, proposals_list):
        """
        Args:
            feature_map    : (B, C, H, W) backbone features
            proposals_list : list of B tensors, each (Ni, 4) in image coords

        Returns:
            pooled : (total_proposals, C, output_size, output_size)
        """
        ######### TODO
        # ---- START YOUR CODE ----
        # Build (total_N, 5) tensor: [batch_idx, x1, y1, x2, y2]
        rois = []
        for batch_idx, proposals in enumerate(proposals_list):
            idx_col = torch.full(
                (proposals.shape[0], 1), batch_idx,
                dtype=proposals.dtype, device=proposals.device
            )
            rois.append(torch.cat([idx_col, proposals], dim=1))  # (Ni, 5)
        rois = torch.cat(rois, dim=0)   # (total_N, 5)

        return self.roi_pool(feature_map, rois)
        # ---- END YOUR CODE ----

### Task 3.5 Detection Head

In [ ]:
class DetectionHead(nn.Module):
    """
    The second-stage detection head.

    Architecture:
        (C, 7, 7) -> flatten -> fc1(C*7*7, 1024) -> ReLU
                                                       |
                                          +------------+------------+
                                          |                         |
                                      cls_score(1024, num_classes)  bbox_pred(1024, num_classes*4)
    """

    def __init__(self, in_features=512 * 7 * 7, num_classes=6):
        super().__init__()
        self.num_classes = num_classes
        ######## TODO
        # ---- START YOUR CODE ----
        self.fc1       = nn.Linear(in_features, 1024)
        self.fc2       = nn.Linear(1024, 1024)
        self.cls_score = nn.Linear(1024, num_classes)
        self.bbox_pred = nn.Linear(1024, num_classes * 4)

        # Weight init
        for layer in [self.fc1, self.fc2]:
            nn.init.normal_(layer.weight, std=0.01)
            nn.init.zeros_(layer.bias)
        nn.init.normal_(self.cls_score.weight, std=0.01)
        nn.init.zeros_(self.cls_score.bias)
        nn.init.normal_(self.bbox_pred.weight, std=0.001)
        nn.init.zeros_(self.bbox_pred.bias)
        # ---- END YOUR CODE ----

    def forward(self, pooled_features):
        """
        Args:
            pooled_features: (N, C, 7, 7) from RoI pooling

        Returns:
            class_logits: (N, num_classes)
            box_deltas  : (N, num_classes * 4)
        """
        ### TODO
        # ---- START YOUR CODE ----
        x = pooled_features.flatten(1)   # (N, C*7*7)
        x = F.relu(self.fc1(x))          # (N, 1024)
        x = F.relu(self.fc2(x))          # (N, 1024)
        class_logits = self.cls_score(x) # (N, num_classes)
        box_deltas   = self.bbox_pred(x) # (N, num_classes * 4)
        return class_logits, box_deltas
        # ---- END YOUR CODE ----

In [ ]:
# ============================================================================
#  BACKBONE (PROVIDED)
# ============================================================================

class Backbone(nn.Module):
    """ResNet-18 backbone that outputs the feature map from layer4."""

    def __init__(self):
        super().__init__()
        resnet = resnet18(weights=ResNet18_Weights.DEFAULT)
        # Use everything up to layer4 (stride 16, 512 channels)
        self.features = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4
        )
        self.out_channels = 512
        self.stride = 16

    def forward(self, x):
        return self.features(x)


# ============================================================================
#  PROPOSAL GENERATION (PROVIDED)
# ============================================================================

def generate_proposals(objectness, box_deltas, anchors, image_size,
                       pre_nms_top_n=6000, post_nms_top_n=300,
                       nms_thresh=0.7, min_size=4):
    """
    Generate proposals from RPN outputs using NMS (provided).

    Steps:
      1. Decode box deltas to get predicted boxes
      2. Clip to image boundaries
      3. Remove tiny boxes
      4. Take top-scoring boxes (pre-NMS)
      5. Apply NMS
      6. Take top boxes (post-NMS)
    """
    # Get objectness scores for "object" class
    scores = objectness[:, 1]  # (A,)

    # Decode boxes
    proposals = decode_boxes(anchors, box_deltas)
    proposals = clip_boxes(proposals, image_size)

    # Filter small boxes
    keep = filter_small_boxes(proposals, min_size)
    proposals = proposals[keep]
    scores = scores[keep]

    # Pre-NMS top-N
    if len(scores) > pre_nms_top_n:
        _, top_idx = scores.topk(pre_nms_top_n)
        proposals = proposals[top_idx]
        scores = scores[top_idx]

    # NMS
    keep = nms(proposals, scores, nms_thresh)

    # Post-NMS top-N
    keep = keep[:post_nms_top_n]
    proposals = proposals[keep]
    scores = scores[keep]

    return proposals.detach(), scores.detach()


# ============================================================================
#  FULL MODEL (PROVIDED — uses your implementations)
# ============================================================================

class SimpleFasterRCNN(nn.Module):
    """
    Simplified Faster R-CNN that ties all pieces together.
    """

    def __init__(self, num_classes=6, image_size=300):
        super().__init__()
        self.num_classes = num_classes
        self.image_size = image_size

        self.backbone = Backbone()
        self.anchor_generator = AnchorGenerator(
            sizes=[64, 128, 256],
            aspect_ratios=[0.5, 1.0, 2.0]
        )
        self.rpn_head = RPNHead(
            in_channels=self.backbone.out_channels,
            num_anchors=self.anchor_generator.num_anchors_per_location
        )
        self.roi_pool = RoIPoolingLayer(
            output_size=7,
            spatial_scale=1.0 / self.backbone.stride
        )
        self.detection_head = DetectionHead(
            in_features=self.backbone.out_channels * 7 * 7,
            num_classes=num_classes
        )

    def forward(self, images, gt_boxes=None, gt_labels=None):
        """
        Args:
            images: (B, 3, H, W)
            gt_boxes: list of (Gi, 4) tensors (training only)
            gt_labels: list of (Gi,) tensors (training only)

        Returns dict with keys depending on mode:
            Training: rpn_cls_loss, rpn_reg_loss, det_cls_loss, det_reg_loss
            Inference: proposals, class_scores, box_deltas
        """
        B = images.shape[0]
        device = images.device

        # Stage 1: Backbone
        features = self.backbone(images)
        feat_h, feat_w = features.shape[2], features.shape[3]

        # Stage 2: RPN
        objectness, rpn_box_deltas = self.rpn_head(features)
        anchors = self.anchor_generator.generate(
            (feat_h, feat_w), self.backbone.stride, device
        )

        # Generate proposals
        all_proposals = []
        for i in range(B):
            proposals, _ = generate_proposals(
                objectness[i], rpn_box_deltas[i], anchors, self.image_size
            )
            all_proposals.append(proposals)

        if self.training and gt_boxes is not None:
            # ---- RPN Loss ----
            total_rpn_cls_loss = 0
            total_rpn_reg_loss = 0
            for i in range(B):
                cls_l, reg_l = rpn_loss(
                    objectness[i], rpn_box_deltas[i], anchors, gt_boxes[i]
                )
                total_rpn_cls_loss += cls_l
                total_rpn_reg_loss += reg_l
            total_rpn_cls_loss /= B
            total_rpn_reg_loss /= B

            # ---- Prepare proposals for second stage ----
            # Add GT boxes to proposals (standard training trick)
            training_proposals = []
            for i in range(B):
                if gt_boxes[i].shape[0] > 0:
                    combined = torch.cat([all_proposals[i], gt_boxes[i]], dim=0)
                else:
                    combined = all_proposals[i]
                training_proposals.append(combined)

            # Sample proposals for detection head training
            sampled_proposals = []
            det_labels = []
            det_reg_targets = []
            for i in range(B):
                props = training_proposals[i]
                if gt_boxes[i].shape[0] == 0 or props.shape[0] == 0:
                    continue

                iou_matrix = compute_iou(props, gt_boxes[i])
                max_iou, matched_gt_idx = iou_matrix.max(dim=1)

                # Assign labels
                prop_labels = torch.zeros(len(props), dtype=torch.long, device=device)
                pos_mask = max_iou >= 0.5
                prop_labels[pos_mask] = gt_labels[i][matched_gt_idx[pos_mask]]

                # Sample 64 proposals: 25% positive, 75% negative
                pos_idx = torch.where(pos_mask)[0]
                neg_idx = torch.where(~pos_mask)[0]
                num_pos = min(16, len(pos_idx))
                num_neg = min(48, len(neg_idx))
                if len(pos_idx) > num_pos:
                    pos_idx = pos_idx[torch.randperm(len(pos_idx))[:num_pos]]
                if len(neg_idx) > num_neg:
                    neg_idx = neg_idx[torch.randperm(len(neg_idx))[:num_neg]]

                keep = torch.cat([pos_idx, neg_idx])
                sampled_proposals.append(props[keep])
                det_labels.append(prop_labels[keep])

                # Regression targets for positives
                reg_t = torch.zeros(len(keep), 4, device=device)
                local_pos = torch.arange(num_pos)
                if num_pos > 0:
                    reg_t[local_pos] = encode_boxes(
                        props[pos_idx],
                        gt_boxes[i][matched_gt_idx[pos_idx]]
                    )
                det_reg_targets.append(reg_t)

            if len(sampled_proposals) == 0:
                return {
                    'rpn_cls_loss': total_rpn_cls_loss,
                    'rpn_reg_loss': total_rpn_reg_loss,
                    'det_cls_loss': torch.tensor(0.0, device=device),
                    'det_reg_loss': torch.tensor(0.0, device=device),
                }

            # ---- Stage 3: RoI Pool + Detection Head ----
            pooled = self.roi_pool(features, sampled_proposals)
            class_logits, box_preds = self.detection_head(pooled)

            all_labels = torch.cat(det_labels)
            all_reg_targets = torch.cat(det_reg_targets)

            det_cls_loss = F.cross_entropy(class_logits, all_labels)

            # Regression loss only on positive proposals
            pos_mask = all_labels > 0
            if pos_mask.sum() > 0:
                pos_labels = all_labels[pos_mask]
                # Get class-specific box predictions
                pos_box_preds = box_preds[pos_mask].view(-1, self.num_classes, 4)
                pos_box_preds = pos_box_preds[torch.arange(len(pos_labels)), pos_labels]
                det_reg_loss = F.smooth_l1_loss(
                    pos_box_preds, all_reg_targets[pos_mask]
                )
            else:
                det_reg_loss = torch.tensor(0.0, device=device)

            return {
                'rpn_cls_loss': total_rpn_cls_loss,
                'rpn_reg_loss': total_rpn_reg_loss,
                'det_cls_loss': det_cls_loss,
                'det_reg_loss': det_reg_loss,
            }

        else:
            # Inference
            if len(all_proposals) > 0 and all(p.shape[0] > 0 for p in all_proposals):
                pooled = self.roi_pool(features, all_proposals)
                class_logits, box_preds = self.detection_head(pooled)
            else:
                class_logits = torch.zeros(0, self.num_classes, device=device)
                box_preds = torch.zeros(0, self.num_classes * 4, device=device)
            return {
                'proposals': all_proposals,
                'class_logits': class_logits,
                'box_deltas': box_preds,
            }



# ============================================================================
#  TRAINING LOOP (PROVIDED)
# ============================================================================

def train(num_epochs=8, batch_size=4, lr=1e-4, image_size=300):
    """Main training loop."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on: {device}")

    dataset = PennFudanDataset(image_size=image_size)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                            collate_fn=collate_fn, num_workers=2)

    model = SimpleFasterRCNN(
        num_classes=PennFudanDataset.NUM_CLASSES,
        image_size=image_size
    ).to(device)

    # Freeze early backbone layers
    for param in model.backbone.features[:5].parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)

    for epoch in range(num_epochs):
        model.train()
        stats = defaultdict(float)
        n_batches = 0

        for batch_idx, (images, gt_boxes, gt_labels) in enumerate(dataloader):
            images = images.to(device)
            gt_boxes = [b.to(device) for b in gt_boxes]
            gt_labels = [l.to(device) for l in gt_labels]

            losses = model(images, gt_boxes, gt_labels)
            total_loss = sum(losses.values())

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            for k, v in losses.items():
                stats[k] += v.item()
            n_batches += 1

            if batch_idx % 50 == 0:
                msg = " | ".join(f"{k}: {v.item():.4f}" for k, v in losses.items())
                print(f"  [{batch_idx}/{len(dataloader)}] {msg}")

        scheduler.step()
        msg = " | ".join(f"{k}: {v/n_batches:.4f}" for k, v in stats.items())
        print(f"Epoch {epoch+1}/{num_epochs}: {msg}")

    return model

In [ ]:
print("Anchor-target assignment...")
viz_anchor_assignment(dataset)

print("Backbone feature maps...")
viz_backbone_features(dataset)

In [ ]:
print("RPN predictions (untrained)...")
viz_rpn_predictions(dataset)

print("Region proposals (untrained)...")
viz_proposals(dataset, device = 'cpu')


print("Detections (untrained)...")
viz_detections(dataset, device = 'cpu')

print("Full pipeline overview...")
viz_pipeline_overview(dataset, device = 'cpu')

In [ ]:
gen = AnchorGenerator()
cell_anchors = gen.generate_cell_anchors()
print(f"\nCell anchors shape: {cell_anchors.shape}")  # (9, 4)
print(f"Cell anchors:\n{cell_anchors}")

anchors = gen.generate((19, 19), stride=16)
print(f"\nTotal anchors for 300x300 image: {anchors.shape}")  # (3249, 4)



In [ ]:
# Part 2: Test RPN head
rpn = RPNHead(in_channels=512, num_anchors=9)
dummy_feat = torch.randn(2, 512, 19, 19)
obj, deltas = rpn(dummy_feat)
print(f"\nRPN Objectness: {obj.shape}")   # (2, 3249, 2)
print(f"RPN Box deltas: {deltas.shape}")  # (2, 3249, 4)

In [ ]:
# Part 3: Test target assignment
gt = torch.tensor([[100, 100, 200, 200]], dtype=torch.float32)
labels, reg_targets = rpn_assign_targets(anchors, gt)
print(f"\nTarget assignment:")
print(f"  Positive anchors: {(labels == 1).sum()}")
print(f"  Negative anchors: {(labels == 0).sum()}")
print(f"  Ignored anchors:  {(labels == -1).sum()}")

In [ ]:
# Part 4: Test RoI pooling
roi_pool = RoIPoolingLayer(output_size=7, spatial_scale=1/16)
feat = torch.randn(1, 512, 19, 19)
props = [torch.tensor([[10, 10, 100, 100], [50, 50, 200, 200]], dtype=torch.float32)]
pooled = roi_pool(feat, props)
print(f"\nRoI pooled features: {pooled.shape}")  # (2, 512, 7, 7)

In [ ]:
# Part 5: Test detection head
det_head = DetectionHead(in_features=512*7*7, num_classes=2)
cls, box = det_head(pooled)
print(f"\nDetection head:")
print(f"  Class logits: {cls.shape}")  # (2, 2)
print(f"  Box deltas: {box.shape}")    # (2, 8)

In [ ]:
# Full model forward
print("\nFull model test:")
model = SimpleFasterRCNN(num_classes=2, image_size=300)
dummy_img = torch.randn(2, 3, 300, 300)
gt_b = [torch.tensor([[50, 50, 150, 150.0]]), torch.tensor([[30, 30, 200, 200.0]])]
gt_l = [torch.tensor([1]), torch.tensor([1])]
model.train()
losses = model(dummy_img, gt_b, gt_l)
for k, v in losses.items():
    print(f"  {k}: {v.item():.4f}")

### Training the network

In [ ]:
model = train(num_epochs=8, batch_size=4)


In [ ]:
viz_proposals(dataset, device = 'cuda', model=model, save_path='viz_07_proposals_trained.png')
viz_detections(dataset, device = 'cuda', model=model, save_path='viz_09_detections_trained.png')
